# 09 - Forecast statistics and calibrated uncertainty (pre-registered Task 4)

1. Paired test of LightGBM vs ETS (static, 726 items, 6 evaluation origins).
2. Forecast bias: pooled and per item, with bootstrap CIs.
3. Split-conformal bands from rolling-origin residuals: category-pooled residuals from earlier origins, scaled by each item's MASE scale, finite-sample corrected. Coverage at 80% and 90%, before and after.
4. Inventory rerun with conformal safety stock (7-day lead time, 90/95/99%), compared with the normal-formula policy of notebook 03. Both fill rate and achieved cycle service level are reported.

In [1]:
import sys
sys.path.append('../src')
import numpy as np
import pandas as pd

from break_detection import weekly_ca_prices
from metrics import paired_test
from pipeline import price_daily_frame, simulate_inventory, summarise_inventory, intermittency

SEED = 42
fc = pd.read_parquet("../data/processed/backtest_forecasts.parquet")
err = pd.read_csv("../data/processed/backtest_errors.csv", parse_dates=['origin'])
scales = err[['item_id', 'origin', 'mase_scale']].drop_duplicates()
ORIGINS = sorted(fc['origin'].unique())
EVAL = ORIGINS[1:]
fc['category'] = fc['item_id'].str.split('_').str[0]
pd.set_option('display.width', 220)
print(len(ORIGINS), "origins")

7 origins


## 1. LightGBM vs ETS, and unrounded MASE

In [2]:
ev = err[err['origin'].isin(EVAL) & (err['variant'] == 'static')]
im = ev.groupby(['item_id', 'category', 'model'], as_index=False)['mase'].mean()
piv = im.pivot_table(index=['item_id', 'category'], columns='model', values='mase').reset_index()
rows = []
for cat in ['ALL', 'FOODS', 'HOBBIES', 'HOUSEHOLD']:
    d = piv if cat == 'ALL' else piv[piv['category'] == cat]
    rows.append({'comparison': 'lightgbm - ets', 'category': cat, **paired_test(d['lightgbm'] - d['ets'], seed=SEED)})
lgb_ets = pd.DataFrame(rows)
lgb_ets.to_csv("../outputs/tables/lgb_vs_ets_test.csv", index=False)
print(lgb_ets[['category', 'n', 'median_diff', 'ci_low', 'ci_high', 'share_improved', 'rank_biserial', 'p_value']].round(5).to_string(index=False))

unr = []
for model in ['seasonal_naive', 'ets', 'lightgbm', 'tsb']:
    d = im[im['model'] == model]
    for cat in ['ALL'] + sorted(d['category'].unique()):
        x = d if cat == 'ALL' else d[d['category'] == cat]
        unr.append({'model': model, 'category': cat, 'n_items': len(x), 'median_item_mase': x['mase'].median(),
                    'mean_item_mase': x['mase'].mean()})
unrounded = pd.DataFrame(unr)
unrounded.to_csv("../outputs/tables/model_mase_unrounded.csv", index=False)
print(unrounded.to_string(index=False))

 category   n  median_diff   ci_low  ci_high  share_improved  rank_biserial  p_value
      ALL 726     -0.00907 -0.01147 -0.00659         0.61570       -0.23627  0.00000
    FOODS 337     -0.01776 -0.02509 -0.01071         0.64985       -0.30957  0.00000
  HOBBIES 106     -0.00488 -0.00974 -0.00199         0.64151       -0.21672  0.05277
HOUSEHOLD 283     -0.00348 -0.00848 -0.00042         0.56537       -0.13472  0.04947
         model  category  n_items  median_item_mase  mean_item_mase
seasonal_naive       ALL      726          0.975636        0.993032
seasonal_naive     FOODS      337          0.956512        0.981944
seasonal_naive   HOBBIES      106          1.004395        1.012656
seasonal_naive HOUSEHOLD      283          0.978060        0.998885
           ets       ALL      726          0.755442        0.784411
           ets     FOODS      337          0.771414        0.805669
           ets   HOBBIES      106          0.755291        0.758126
           ets HOUSEHOLD      2

## 2. Bias

In [3]:
rng = np.random.default_rng(SEED)
POL_BIAS = [('seasonal_naive', 'static'), ('ets', 'static'), ('lightgbm', 'static'), ('lightgbm', 'adaptive')]
e = fc[fc['origin'].isin(EVAL)]
rows = []
for model, variant in POL_BIAS:
    d = e[(e['model'] == model) & (e['variant'] == variant)]
    for cat in ['ALL', 'FOODS', 'HOBBIES', 'HOUSEHOLD']:
        x = d if cat == 'ALL' else d[d['category'] == cat]
        per = x.groupby('item_id')[['forecast', 'actual']].sum()
        per = per[per['actual'] > 0]
        b = (per['forecast'] / per['actual'] - 1).to_numpy()
        boots = np.median(rng.choice(b, size=(2000, len(b))), axis=1)
        rows.append({'model': model, 'variant': variant, 'category': cat, 'n_items': len(b),
                     'pooled_bias': x['forecast'].sum() / x['actual'].sum() - 1,
                     'median_item_bias': np.median(b), 'ci_low': np.quantile(boots, 0.025), 'ci_high': np.quantile(boots, 0.975),
                     'share_items_under': np.mean(b < 0)})
bias = pd.DataFrame(rows)
bias.to_csv("../outputs/tables/forecast_bias.csv", index=False)
print(bias.round(4).to_string(index=False))

         model  variant  category  n_items  pooled_bias  median_item_bias  ci_low  ci_high  share_items_under
seasonal_naive   static       ALL      726      -0.0078            0.0014 -0.0052   0.0083             0.4904
seasonal_naive   static     FOODS      337      -0.0253           -0.0200 -0.0317  -0.0115             0.6261
seasonal_naive   static   HOBBIES      106       0.0451            0.0390  0.0183   0.0616             0.2642
seasonal_naive   static HOUSEHOLD      283       0.0149            0.0147  0.0046   0.0274             0.4134
           ets   static       ALL      726      -0.0088           -0.0073 -0.0133  -0.0011             0.5399
           ets   static     FOODS      337      -0.0193           -0.0216 -0.0313  -0.0119             0.5905
           ets   static   HOBBIES      106       0.0164            0.0148 -0.0010   0.0271             0.4151
           ets   static HOUSEHOLD      283       0.0073           -0.0023 -0.0114   0.0059             0.5265
      ligh

## 3. Conformal bands
**Before:** the existing method (each item's own empirical error quantiles from earlier origins).
**After:** category-pooled normalized residuals from earlier origins (residual ÷ the item's MASE scale at that origin), with the finite-sample correction ⌈(n+1)(1−α/2)⌉/n.

In [4]:
POL = [('lightgbm', 'static'), ('lightgbm', 'adaptive'), ('seasonal_naive', 'static')]
f = fc[fc.set_index(['model', 'variant']).index.isin(POL)].merge(scales, on=['item_id', 'origin'])
f['e'] = f['actual'] - f['forecast']
f['r'] = f['e'] / f['mase_scale']

def conformal_offsets(r, alpha):
    r = np.asarray(r)
    n = len(r)
    level = min(1.0, np.ceil((n + 1) * (1 - alpha / 2)) / n)
    return -np.quantile(-r, level, method='higher'), np.quantile(r, level, method='higher')

rows = []
for (model, variant), g in f.groupby(['model', 'variant']):
    for k, origin in enumerate(EVAL, start=1):
        test = g[g['origin'] == origin]
        for alpha in [0.2, 0.1]:
            lvl = int(round(100 * (1 - alpha)))
            # before: per-item empirical quantiles of raw errors from earlier origins
            past = g[g['origin'] < origin].groupby('item_id')['e']
            q = past.quantile([alpha / 2, 1 - alpha / 2]).unstack()
            t = test.join(q, on='item_id')
            lo = np.maximum(t['forecast'] + t[alpha / 2], 0)
            hi = t['forecast'] + t[1 - alpha / 2]
            inside_b = (t['actual'] >= lo) & (t['actual'] <= hi)
            # after: category-pooled normalized conformal
            inside_a = pd.Series(False, index=test.index)
            width_a = pd.Series(np.nan, index=test.index)
            for cat, tc in test.groupby('category'):
                pool = g[(g['origin'] < origin) & (g['category'] == cat)]['r']
                qlo, qhi = conformal_offsets(pool, alpha)
                lo_a = np.maximum(tc['forecast'] + qlo * tc['mase_scale'], 0)
                hi_a = tc['forecast'] + qhi * tc['mase_scale']
                inside_a.loc[tc.index] = (tc['actual'] >= lo_a) & (tc['actual'] <= hi_a)
                width_a.loc[tc.index] = hi_a - lo_a
            rows.append({'model': model, 'variant': variant, 'origin_index': k, 'origin': origin, 'level': lvl,
                         'n_days': len(test), 'coverage_before': inside_b.mean(), 'coverage_after': inside_a.mean(),
                         'mean_width_before': (hi - lo).mean(), 'mean_width_after': width_a.mean()})
cov = pd.DataFrame(rows)
cov.to_csv("../outputs/tables/conformal_coverage_by_origin.csv", index=False)
cov['origins'] = np.where(cov['origin_index'] == 1, 'origin 1 (warm-up only)', 'origins 2-6')
cs = cov.groupby(['model', 'variant', 'level', 'origins']).apply(
    lambda d: pd.Series({'n_days': d['n_days'].sum(),
                         'coverage_before': np.average(d['coverage_before'], weights=d['n_days']),
                         'coverage_after': np.average(d['coverage_after'], weights=d['n_days']),
                         'mean_width_before': np.average(d['mean_width_before'], weights=d['n_days']),
                         'mean_width_after': np.average(d['mean_width_after'], weights=d['n_days'])}),
    include_groups=False).reset_index()
cs['within_2pts_after'] = (cs['coverage_after'] - cs['level'] / 100).abs() <= 0.02
cs.to_csv("../outputs/tables/conformal_coverage.csv", index=False)
print(cs.round(4).to_string(index=False))

         model  variant  level                 origins   n_days  coverage_before  coverage_after  mean_width_before  mean_width_after  within_2pts_after
      lightgbm adaptive     80 origin 1 (warm-up only)  20328.0           0.6772          0.7962            13.0160           15.6280               True
      lightgbm adaptive     80             origins 2-6 101640.0           0.7554          0.7970            15.3243           16.6625               True
      lightgbm adaptive     90 origin 1 (warm-up only)  20328.0           0.7839          0.8920            16.5128           20.7954               True
      lightgbm adaptive     90             origins 2-6 101640.0           0.8532          0.8976            19.8344           22.4348               True
      lightgbm   static     80 origin 1 (warm-up only)  20328.0           0.6901          0.7948            12.9645           15.5653               True
      lightgbm   static     80             origins 2-6 101640.0           0.7561  

## 4. Inventory with conformal safety stock
Safety stock = item scale × the service-level quantile (finite-sample corrected, upper side) of normalized 7-day error sums from earlier origins, pooled within category and policy. First, a check that the reusable simulation reproduces notebook 03 with the normal rule.

In [5]:
long_df = pd.concat([pd.read_csv(f"../data/interim/{c}_candidates.csv", parse_dates=['date'])
                     for c in ['foods', 'hobbies', 'household']], ignore_index=True)
calendar = pd.read_csv("../data/raw/calendar.csv", parse_dates=['date'])
prices = pd.read_csv("../data/raw/sell_prices.csv", dtype={'store_id': 'category'})
price_wk = weekly_ca_prices(prices, calendar, long_df['item_id'].unique())
del prices
days = pd.date_range(EVAL[0] + pd.Timedelta(days=1), periods=28 * len(EVAL))
price_daily = price_daily_frame(price_wk, days)
inter = intermittency(long_df, ORIGINS[0])
L, SLS = 7, [0.90, 0.95, 0.99]

normal = simulate_inventory(fc, ORIGINS, price_daily, POL, SLS, L, inter)
ns = summarise_inventory(normal)
ref = pd.read_csv("../outputs/tables/inventory_policy_summary.csv")
chk = ns.merge(ref, on=['policy', 'service_level'], suffixes=('', '_nb03'))
print("max |total cost - notebook 03|:", np.abs(chk['total_cost'] - chk['total_cost_nb03']).max().round(6),
      "| max |fill rate - notebook 03| (nb03 rounded to 3 dp):", np.abs(chk['fill_rate'] - chk['fill_rate_nb03']).max().round(6))

max |total cost - notebook 03|: 0.000455 | max |fill rate - notebook 03| (nb03 rounded to 3 dp): 0.000389


In [6]:
sums = []
for (item, origin, model, variant, cat), g in f.groupby(['item_id', 'origin', 'model', 'variant', 'category']):
    s = np.convolve(g.sort_values('date')['e'].to_numpy(), np.ones(L), mode='valid') / g['mase_scale'].iloc[0]
    sums.append(pd.DataFrame({'origin': origin, 'model': model, 'variant': variant, 'category': cat, 'r7': s}))
sums = pd.concat(sums, ignore_index=True)
scale_map = scales.set_index(['item_id', 'origin'])['mase_scale']
cat_map = long_df.groupby('item_id')['item_id'].first().str.split('_').str[0]
qcache = {}

def conformal_ss(item, model, variant, origin, lead_time, sl, past_errors):
    cat = cat_map[item]
    key = (cat, model, variant, origin, sl)
    if key not in qcache:
        pool = sums[(sums['category'] == cat) & (sums['model'] == model) & (sums['variant'] == variant)
                    & (sums['origin'] < origin)]['r7'].to_numpy()
        n = len(pool)
        qcache[key] = np.quantile(pool, min(1.0, np.ceil((n + 1) * sl) / n), method='higher')
    return max(0.0, scale_map[(item, origin)] * qcache[key])

conf = simulate_inventory(fc, ORIGINS, price_daily, POL, SLS, L, inter, ss_fn=conformal_ss)
both = pd.concat([normal.assign(method='normal formula'), conf.assign(method='conformal')], ignore_index=True)
both.to_csv("../outputs/tables/inventory_calibrated_items.csv", index=False)
inv = summarise_inventory(both, by=('method', 'policy', 'service_level'))
inv.to_csv("../outputs/tables/inventory_calibrated_vs_normal.csv", index=False)
print(inv[['method', 'policy', 'service_level', 'fill_rate', 'cycle_service_level', 'cycles', 'stockout_days',
           'avg_inventory_value', 'holding_cost', 'stockout_cost', 'total_cost']].round(4).to_string(index=False))

        method                policy  service_level  fill_rate  cycle_service_level  cycles  stockout_days  avg_inventory_value  holding_cost  stockout_cost  total_cost
     conformal     lightgbm_adaptive           0.90     0.9793               0.8585   15293           2162          173437.5904    19957.2022     50321.1331  70278.3353
     conformal     lightgbm_adaptive           0.95     0.9872               0.9112   15436           1234          199487.9230    22954.7747     32869.0250  55823.7997
     conformal     lightgbm_adaptive           0.99     0.9961               0.9698   15618            334          272448.2318    31350.2075     10982.9842  42333.1917
     conformal       lightgbm_static           0.90     0.9831               0.8653   15216           1970          170766.7714    19649.8751     40482.9408  60132.8159
     conformal       lightgbm_static           0.95     0.9900               0.9165   15345           1135          195327.4102    22476.0308     24829.510

In [7]:
rows = []
for (model, variant), _ in both.groupby(['model', 'variant']):
    for sl in SLS:
        p = both[(both['model'] == model) & (both['variant'] == variant) & (both['service_level'] == sl)]
        w = p.pivot(index='item_id', columns='method', values='total_cost')
        rows.append({'policy': f'{model}_{variant}', 'service_level': sl,
                     **paired_test(w['conformal'] - w['normal formula'], seed=SEED)})
cmp_ = pd.DataFrame(rows)
cmp_.to_csv("../outputs/tables/inventory_calibrated_paired.csv", index=False)
print("Per-item total cost, conformal - normal formula:")
print(cmp_[['policy', 'service_level', 'n', 'median_diff', 'ci_low', 'ci_high', 'share_improved', 'p_value']].round(4).to_string(index=False))

Per-item total cost, conformal - normal formula:
               policy  service_level   n  median_diff   ci_low  ci_high  share_improved  p_value
    lightgbm_adaptive           0.90 726     -12.0439 -13.6935 -10.3824          0.7769   0.0000
    lightgbm_adaptive           0.95 726      -8.2848 -10.0326  -6.5920          0.6901   0.0000
    lightgbm_adaptive           0.99 726       2.4520   1.5083   3.5947          0.4174   0.3902
      lightgbm_static           0.90 726     -12.6100 -14.2554 -10.5189          0.7906   0.0000
      lightgbm_static           0.95 726      -8.4905 -10.0923  -6.7893          0.7190   0.0000
      lightgbm_static           0.99 726       2.5610   1.5856   3.6056          0.4187   0.1639
seasonal_naive_static           0.90 726      -0.7355  -1.2329  -0.2748          0.5579   0.0000
seasonal_naive_static           0.95 726      -1.1203  -1.9078  -0.5012          0.5909   0.0000
seasonal_naive_static           0.99 726       1.8271   1.3457   2.5610       

## 5. Item-level tables for the dashboard
Paired tests for every model pair, each item's forecast bias, and each item's band coverage before and after conformal calibration (origins 2-6). Nothing new is modelled; these re-express the results above per item.

In [8]:
rows = []
for a, b in [('lightgbm', 'seasonal_naive'), ('lightgbm', 'ets'), ('ets', 'seasonal_naive')]:
    for cat in ['ALL', 'FOODS', 'HOBBIES', 'HOUSEHOLD']:
        d = piv if cat == 'ALL' else piv[piv['category'] == cat]
        rows.append({'model_a': a, 'model_b': b, 'category': cat, **paired_test(d[a] - d[b], seed=SEED)})
pairwise = pd.DataFrame(rows)
pairwise.to_csv("../outputs/tables/model_pairwise_tests.csv", index=False)
print(pairwise[['model_a', 'model_b', 'category', 'n', 'median_diff', 'ci_low', 'ci_high', 'share_improved', 'p_value']].round(5).to_string(index=False))

item_bias = []
for model, variant in POL_BIAS:
    d = e[(e['model'] == model) & (e['variant'] == variant)].groupby('item_id')[['forecast', 'actual']].sum()
    item_bias.append(pd.DataFrame({'item_id': d.index, 'model': model, 'variant': variant,
                                   'forecast_units': d['forecast'].to_numpy(), 'actual_units': d['actual'].to_numpy(),
                                   'bias': np.where(d['actual'] > 0, d['forecast'] / d['actual'] - 1, np.nan)}))
item_bias = pd.concat(item_bias, ignore_index=True)
item_bias.to_csv("../outputs/tables/item_bias.csv", index=False)
print(item_bias.groupby(['model', 'variant'])['bias'].median().round(4).to_string())

 model_a        model_b  category   n  median_diff   ci_low  ci_high  share_improved  p_value
lightgbm seasonal_naive       ALL 726     -0.22555 -0.23430 -0.21465         0.93526  0.00000
lightgbm seasonal_naive     FOODS 337     -0.20323 -0.21680 -0.19125         0.90208  0.00000
lightgbm seasonal_naive   HOBBIES 106     -0.26335 -0.28499 -0.23729         0.97170  0.00000
lightgbm seasonal_naive HOUSEHOLD 283     -0.23290 -0.24337 -0.21806         0.96113  0.00000
lightgbm            ets       ALL 726     -0.00907 -0.01147 -0.00659         0.61570  0.00000
lightgbm            ets     FOODS 337     -0.01776 -0.02509 -0.01071         0.64985  0.00000
lightgbm            ets   HOBBIES 106     -0.00488 -0.00974 -0.00199         0.64151  0.05277
lightgbm            ets HOUSEHOLD 283     -0.00348 -0.00848 -0.00042         0.56537  0.04947
     ets seasonal_naive       ALL 726     -0.21026 -0.21916 -0.19873         0.96006  0.00000
     ets seasonal_naive     FOODS 337     -0.17283 -0.18629 

model           variant 
ets             static     -0.0073
lightgbm        adaptive   -0.0193
                static     -0.0248
seasonal_naive  static      0.0014


In [9]:
rows = []
for (model, variant), g in f.groupby(['model', 'variant']):
    for origin in EVAL[1:]:
        test = g[g['origin'] == origin]
        past = g[g['origin'] < origin]
        q = past.groupby('item_id')['e'].quantile([0.05, 0.95]).unstack()
        t = test.join(q, on='item_id')
        before = (t['actual'] >= np.maximum(t['forecast'] + t[0.05], 0)) & (t['actual'] <= t['forecast'] + t[0.95])
        after = pd.Series(False, index=test.index)
        for cat, tc in test.groupby('category'):
            qlo, qhi = conformal_offsets(past[past['category'] == cat]['r'], 0.1)
            after.loc[tc.index] = (tc['actual'] >= np.maximum(tc['forecast'] + qlo * tc['mase_scale'], 0)) & \
                                  (tc['actual'] <= tc['forecast'] + qhi * tc['mase_scale'])
        rows.append(pd.DataFrame({'item_id': test['item_id'], 'model': model, 'variant': variant,
                                  'inside_before': before.to_numpy(), 'inside_after': after.to_numpy()}))
cov_items = pd.concat(rows).groupby(['item_id', 'model', 'variant'], as_index=False).agg(
    n_days=('inside_before', 'size'), coverage90_before=('inside_before', 'mean'), coverage90_after=('inside_after', 'mean'))
cov_items.to_csv("../outputs/tables/item_band_coverage.csv", index=False)
chk = cov_items[(cov_items['model'] == 'lightgbm') & (cov_items['variant'] == 'static')]
print("LightGBM static 90% coverage, origins 2-6 (should match section 3):",
      round(np.average(chk['coverage90_before'], weights=chk['n_days']), 4), round(np.average(chk['coverage90_after'], weights=chk['n_days']), 4))

LightGBM static 90% coverage, origins 2-6 (should match section 3): 0.8555 0.8992
